In [1]:


#%% 
!pip install --no-index --find-links=/kaggle/input/tf-efficientnet-whl-files /kaggle/input/tf-efficientnet-whl-files/efficientnet-1.1.1-py3-none-any.whl
!pip install --no-index --find-links=/kaggle/input/cupy20240401 cupy
PLATFORM = 'kaggle' # local kaggle
NEEDTRAIN = False
DATATYPE = ['eeg', 'spe', 'img'] # 'eeg', 'spe', 'img', 'stft'
STAGETRAIN = [2, 3]
STAGETEST = 3
print(DATATYPE)
LOAD_MODELS_FROM = 'models2024040501'
if PLATFORM == 'local':
    LOAD_MODELS_FROM = f'./input/{LOAD_MODELS_FROM}'
elif PLATFORM == 'kaggle':
    LOAD_MODELS_FROM = f'/kaggle/input/{LOAD_MODELS_FROM}'

threshold = 0.2

EEG_LENGTH = 30 # s
SFREQ = 100

HIGH = 128 # 128
LENGTH = 256 # 256

IMG_HIGH = 64
IMG_WIDE = 256

SEED = 2024

NSPLIT = 5

BATCHSIZE = 16

READ_SPEC_FILES = False
READ_EEG_FILES = False
READ_IMG_FILES = False
READ_STFT_FILES = False

READ_EXTRA_SPEC_FILES = True
READ_EXTRA_EEG_FILES = True
READ_EXTRA_IMG_FILES = True
READ_EXTRA_STFT_FILES = True

spectrograms = {}
eegs = {}
imgs = {}
stfts = {}

spectrograms2 = {}
eegs2 = {}
imgs2 = {}
stfts2 = {}

# 0.5 -> 0.1
filter_range = [0.5, 40]

BRAIN = {'LL': ['Fp1-F7', 'F7-T3', 'T3-T5', 'T5-O1'],
         'RL': ['Fp2-F8', 'F8-T4', 'T4-T6', 'T6-O2'],
         'LP': ['Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1'],
         'RP': ['Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2']}


import os
import io
from PIL import Image
os.environ["CUDA_VISIBLE_DEVICES"]="0, 1"
import tensorflow as tf
# from einops.layers.tensorflow import Rearrange
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tensorflow.python.framework.ops import reset_default_graph
from sklearn.metrics import confusion_matrix



import cupy as cp







print('TensorFlow version =',tf.__version__)

# USE MULTIPLE GPUS
gpus = tf.config.list_physical_devices('GPU')
if len(gpus)<=1: 
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    print(f'Using {len(gpus)} GPU')
else: 
    strategy = tf.distribute.MirroredStrategy()
    print(f'Using {len(gpus)} GPUs')

VER = 1


np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()





#%% USE MIXED PRECISION
MIX = True
if MIX:
    tf.config.optimizer.set_experimental_options({"auto_mixed_precision": True})
    print('Mixed precision enabled')
else:
    print('Using full precision')
    



#%% Load Train Data
if PLATFORM == 'local':
    df = pd.read_csv('./input/hms-harmful-brain-activity-classification/train.csv')
elif PLATFORM == 'kaggle':
    df = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/train.csv')
TARGETS = df.columns[-6:]
print('Train shape:', df.shape )
print('Targets', list(TARGETS))
df.head()





#%% Create Non-Overlapping EEG Id Train Data
if NEEDTRAIN:
    TARGETS_RAW = list()
    for i in TARGETS:
        TARGETS_RAW.append(i + '_raw')
        
    if READ_SPEC_FILES * READ_EEG_FILES * READ_IMG_FILES * READ_STFT_FILES:
        train_max = df.groupby('eeg_id')[['eeg_label_offset_seconds']].agg({'eeg_label_offset_seconds': 'max'})
        train_min = df.groupby('eeg_id')[['eeg_label_offset_seconds']].agg({'eeg_label_offset_seconds': 'min'})
        
        train_max.columns = ['eeg_label_offset_seconds_max']
        train_min.columns = ['eeg_label_offset_seconds_min']
        
        df2 = df.merge(train_max, on='eeg_id')
        df2 = df2.merge(train_min, on='eeg_id')
        
        df2['s_max'] = abs(df2.eeg_label_offset_seconds - df2.eeg_label_offset_seconds_max)
        df2['s_min'] = abs(df2.eeg_label_offset_seconds - df2.eeg_label_offset_seconds_min)
        
        xx = df2.loc[:, ['s_max', 's_min']].min(1)
        df2 = df2.iloc[:, :15]
        df2['selected'] = xx
        
        df2 = df2.sort_values('selected', ascending=False).reset_index(drop=True)
        df3 = df2.drop_duplicates('eeg_id').reset_index(drop=True)
        
        num_all = 0
        for i in range(len(TARGETS)):
            num_all = max(num_all, sum(np.argmax(df3[TARGETS].values, 1) == i))
        
        train = pd.DataFrame()
        # for i in range(len(TARGETS)):
        #     train_temp = df2.iloc[np.argmax(df2[TARGETS].values, 1) == i].reset_index(drop=True)
        #     train_temp = train_temp.iloc[np.random.permutation(len(train_temp))].reset_index(drop=True)
        #     ii = 1
        #     while len(train_temp.groupby('eeg_id').head(ii)) < num_all:
        #         ii = ii + 1
        #     print(ii)
        #     if ii > 1:
        #         if len(train_temp.groupby('eeg_id').head(ii)) > num_all:
        #             train_temp1 = train_temp.groupby('eeg_id').head(ii-1)
        #             train_temp2 = train_temp.groupby('eeg_id').head(ii)
        #             train_temp2 = pd.concat((train_temp1, train_temp2)).reset_index(drop=True)
        #             train_temp2 = train_temp2.drop_duplicates(keep=False).reset_index(drop=True)
        #             train_temp2 = train_temp2.iloc[np.random.permutation(len(train_temp2))].reset_index(drop=True)
        #             train_temp2 = train_temp2[:(num_all - len(train_temp.groupby('eeg_id').head(ii-1)))]
        #             train_temp = pd.concat((train_temp1, train_temp2)).reset_index(drop=True)
        #     else:
        #         train_temp = train_temp.groupby('eeg_id').head(ii)
        #     print(len(train_temp))
        #     train = pd.concat([train, train_temp]).reset_index(drop=True)
        train = pd.concat([train, df3]).reset_index(drop=True)
        train['sign_id'] = train.index.values
        
        y_data = train[TARGETS].values
        train[TARGETS_RAW] = y_data
        y_data = y_data / y_data.sum(axis=1,keepdims=True)
        train[TARGETS] = y_data
        
        train.head()
        
        train.to_csv('train.csv', index=False)
    else:
        train = pd.read_csv('train.csv')











#%% Read Train Spectrograms
if NEEDTRAIN:
    # READ ALL SPECTROGRAMS
    if PLATFORM == 'local':
        PATH = './input/hms-harmful-brain-activity-classification/train_spectrograms/'
    elif PLATFORM == 'kaggle':
        PATH = '/kaggle/input/hms-harmful-brain-activity-classification/train_spectrograms/'
    files = os.listdir(PATH)
    print(f'There are {len(files)} spectrogram parquets')
    if READ_SPEC_FILES:    
        spectrograms = {}
        for i, f in enumerate(files):
            if i%100==0: print(i,', ',end='')
            tmp = pd.read_parquet(f'{PATH}{f}')
            name = int(f.split('.')[0])
            spectrograms[name] = tmp.iloc[:,1:].values
        if not os.path.exists('./input/brain-spectrograms'):
            os.makedirs('./input/brain-spectrograms')
        np.save('./input/brain-spectrograms/specs.npy', spectrograms, allow_pickle=True)
    else:
        if 'spe' in DATATYPE:
            if PLATFORM == 'local':
                spectrograms = np.load('./input/brain-spectrograms/specs.npy', allow_pickle=True).item()
            elif PLATFORM == 'kaggle':
                spectrograms = np.load('/kaggle/input/brain-spectrograms/specs.npy', allow_pickle=True).item()
        
        
        
        
#%% Read Train EEGs
if NEEDTRAIN:
    
    from scipy import signal
    import time
    
    # READ ALL SPECTROGRAMS
    if PLATFORM == 'local':
        PATH = './input/hms-harmful-brain-activity-classification/train_eegs/'
    elif PLATFORM == 'kaggle':
        PATH = '/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/'
    
    if READ_EEG_FILES or READ_IMG_FILES or READ_STFT_FILES:    
        eegs = {}
        imgs = {}
        stfts = {}
        b, a = signal.butter(3, np.float32(filter_range)*2/SFREQ, 'bandpass')
        
        time_start_jishi = time.time()
        
        for i in range(len(train)):
            if i%100==0:
                xx = (time.time() - time_start_jishi)
                yy = xx / (i+1) * len(train)
                xx = round(xx / 60 * 100) / 100
                yy = round(yy / 60 * 100) / 100
                print(i, f'time: {xx} min / {yy} min')
            eeg_default = pd.read_parquet(os.path.join(PATH, (str(train.eeg_id[i]) + '.parquet')))
            
            list_eeg = list()
            list_img = list()
            list_stft = list()
            for region in BRAIN.keys():
    
                eeg = np.zeros((len(BRAIN[region]), eeg_default.shape[0]), dtype=np.float32)
                for chan_i, chan in enumerate(BRAIN[region]):
                    eeg[chan_i, :] = (eeg_default.loc[:, chan.split('-')[0]] - eeg_default.loc[:, chan.split('-')[1]]).values
                
                eeg[np.isnan(eeg)] = 0
                
                if 200 != SFREQ:
                    eeg = signal.resample_poly(eeg, SFREQ, 200, axis=1)
    
    
    
                eeg = signal.filtfilt(b, a, eeg, axis=1)



                time_temp  = train.eeg_label_offset_seconds[i]
                time_start = round(time_temp * SFREQ + (50 - EEG_LENGTH) / 2 * SFREQ) 
                time_stop  = round(time_temp * SFREQ + (50 + EEG_LENGTH) / 2 * SFREQ)



                list_img.append(eeg[:, time_start:time_stop])
                
                
                
                
                # GPU-accelerated spectrogram computation
                frequencies, times, Sxx = cp._cupyx.scipy.signal.spectrogram(eeg[:, round(time_temp*SFREQ):round((time_temp+50)*SFREQ)],
                                                            SFREQ, nperseg=256, noverlap=219,
                                                            nfft=320)
                valid_freq = (frequencies > 0.0) & (frequencies <= 20)
                Sxx_filtered = Sxx[:, valid_freq, :-1]
    
                
                # spectrogram_slice = cp.clip(Sxx_filtered, cp.exp(-4), cp.exp(6))
                # spectrogram_slice = cp.log(spectrogram_slice)
                # normalization_epsilon = 1e-6
                # mean = spectrogram_slice.mean(axis=(0, 1), keepdims=True)
                # std = spectrogram_slice.std(axis=(0, 1), keepdims=True)
                # spectrogram_slice = (spectrogram_slice - mean) / (std + normalization_epsilon)
                Sxx_filtered = np.reshape(Sxx_filtered.get(), (Sxx_filtered.shape[0],
                                                               Sxx_filtered.shape[1],
                                                               Sxx_filtered.shape[2],
                                                               1))
    
                list_stft.append(Sxx_filtered)




                
                
                list_eeg.append(np.reshape(eeg, (eeg.shape[0], eeg.shape[1], 1)))
                
                
                
                
            list_eeg = np.concatenate(list_eeg, 2)
            
            list_stft = np.concatenate(list_stft, -1)
            
            
            if READ_EEG_FILES:
                
                if not train.eeg_id[i] in eegs.keys():
                    list_eeg = np.array(list_eeg, dtype=np.float32)
                    eegs[train.eeg_id[i]] = list_eeg
            
            if READ_IMG_FILES:
                eeg_all_region = np.concatenate(list_img, 0)
                
                
                fig = plt.figure(clear=True)
                fig.patch.set_facecolor('black')
                amp = 200
                for ii in range(eeg_all_region.shape[0]):
                    jj = ii * amp + (ii // 4) * amp
                    plt.plot(eeg_all_region[ii, :] + jj, color='red', linewidth=0.5)
                plt.xlim(-10, eeg_all_region.shape[1] + 10)
                plt.ylim(-amp/2, eeg_all_region.shape[0] * amp + amp/2*5)
                plt.axis('off')


                byte_stream = io.BytesIO()
                plt.savefig(byte_stream, format='png', bbox_inches='tight')
                byte_stream.seek(0)
                img = Image.open(byte_stream)
                img = np.array(img)[:, :, :1]
                byte_stream.truncate()
                plt.close('all')
    
    
                img = np.concatenate((img, img, img), 2)
                img = np.array(tf.image.resize(img/255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32)
                img = img[:, :, 0:1]
                
                plt.imshow(img), train.expert_consensus[i], i
                
                img = np.concatenate([img[0*IMG_HIGH:1*IMG_HIGH, :, :],
                                      img[1*IMG_HIGH:2*IMG_HIGH, :, :],
                                      img[2*IMG_HIGH:3*IMG_HIGH, :, :],
                                      img[3*IMG_HIGH:4*IMG_HIGH, :, :]], -1)
                
                img[:, :, 0] = -img[:, :, 0]
                img[:, :, 2] = -img[:, :, 2]
                
                img = np.reshape(img, (img.shape[0], img.shape[1], img.shape[2], 1))
                
                
                
                
                
                
                
                
                
                
                eeg_all_region2 = eeg_all_region[:, round(eeg_all_region.shape[1]*1/4):round(eeg_all_region.shape[1]*3/4)]
                fig = plt.figure(clear=True)
                fig.patch.set_facecolor('black')
                amp = 150
                for ii in range(eeg_all_region2.shape[0]):
                    jj = ii * amp + (ii // 4) * amp
                    plt.plot(eeg_all_region2[ii, :] + jj, color='red', linewidth=0.5)
                plt.xlim(-5, eeg_all_region2.shape[1] + 5)
                plt.ylim(-amp/2, eeg_all_region2.shape[0] * amp + amp/2*5)
                plt.axis('off')
                
                
                byte_stream = io.BytesIO()
                plt.savefig(byte_stream, format='png', bbox_inches='tight')
                byte_stream.seek(0)
                img2 = Image.open(byte_stream)
                img2 = np.array(img2)[:, :, :1]
                byte_stream.truncate()
                plt.close('all')
    
    
                img2 = np.concatenate((img2, img2, img2), 2)
                img2 = np.array(tf.image.resize(img2/255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32)
                img2 = img2[:, :, 0:1]
                
    
                img2 = np.concatenate([img2[0*IMG_HIGH:1*IMG_HIGH, :, :],
                                      img2[1*IMG_HIGH:2*IMG_HIGH, :, :],
                                      img2[2*IMG_HIGH:3*IMG_HIGH, :, :],
                                      img2[3*IMG_HIGH:4*IMG_HIGH, :, :]], -1)
                
                img2[:, :, 0] = -img2[:, :, 0]
                img2[:, :, 2] = -img2[:, :, 2]
                
                
                img2 = np.reshape(img2, (img2.shape[0], img2.shape[1], img2.shape[2], 1))
                
                
                
                
                
                
                eeg_all_region3 = eeg_all_region[:, round(eeg_all_region.shape[1]*2/5):round(eeg_all_region.shape[1]*3/5)]
                fig = plt.figure(clear=True)
                fig.patch.set_facecolor('black')
                amp = 100
                for ii in range(eeg_all_region3.shape[0]):
                    jj = ii * amp + (ii // 4) * amp
                    plt.plot(eeg_all_region3[ii, :] + jj, color='red', linewidth=0.5)
                plt.xlim(-2, eeg_all_region3.shape[1] + 2)
                plt.ylim(-amp/2, eeg_all_region3.shape[0] * amp + amp/2*5)
                plt.axis('off')
                
                
                byte_stream = io.BytesIO()
                plt.savefig(byte_stream, format='png', bbox_inches='tight')
                byte_stream.seek(0)
                img3 = Image.open(byte_stream)
                img3 = np.array(img3)[:, :, :1]
                byte_stream.truncate()
                plt.close('all')
    
    
                img3 = np.concatenate((img3, img3, img3), 2)
                img3 = np.array(tf.image.resize(img3/255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32)
                img3 = img3[:, :, 0:1]
                
    
                img3 = np.concatenate([img3[0*IMG_HIGH:1*IMG_HIGH, :, :],
                                      img3[1*IMG_HIGH:2*IMG_HIGH, :, :],
                                      img3[2*IMG_HIGH:3*IMG_HIGH, :, :],
                                      img3[3*IMG_HIGH:4*IMG_HIGH, :, :]], -1)
                
                img3[:, :, 0] = -img3[:, :, 0]
                img3[:, :, 2] = -img3[:, :, 2]
                
                img3 = np.reshape(img3, (img3.shape[0], img3.shape[1], img3.shape[2], 1))
                
                
                
                img = np.concatenate([img, img2, img3], -1)

                imgs[train.sign_id[i]] = img
            
            
            
            if READ_STFT_FILES:
                list_stft = np.array(list_stft, dtype=np.float32)
                stfts[train.sign_id[i]] = list_stft

        
        if READ_EEG_FILES:
            if not os.path.exists('./input/brain-eegs'):
                os.makedirs('./input/brain-eegs')
            np.save('./input/brain-eegs/eegs.npy', eegs, allow_pickle=True)
        if READ_IMG_FILES:
            if not os.path.exists('./input/brain-imgs'):
                os.makedirs('./input/brain-imgs')
            np.save('./input/brain-imgs/imgs.npy', imgs, allow_pickle=True)
        if READ_STFT_FILES:
            if not os.path.exists('./input/brain-stfts'):
                os.makedirs('./input/brain-stfts')
            np.save('./input/brain-stfts/stfts.npy', stfts, allow_pickle=True)

    else:
        if PLATFORM == 'local':
            if 'eeg' in DATATYPE:
                eegs = np.load('./input/brain-eegs/eegs.npy', allow_pickle=True).item()
            if 'img' in DATATYPE:
                imgs = np.load('./input/brain-imgs/imgs.npy', allow_pickle=True).item()
            if 'stft' in DATATYPE:
                stfts = np.load('./input/brain-stfts/stfts.npy', allow_pickle=True).item()
        elif PLATFORM == 'kaggle':
            if 'eeg' in DATATYPE:
                eegs = np.load('/kaggle/input/brain-eegs/eegs.npy', allow_pickle=True).item()
            if 'img' in DATATYPE:
                imgs = np.load('/kaggle/input/brain-imgs/imgs.npy', allow_pickle=True).item()
            if 'stft' in DATATYPE:
                stfts = np.load('/kaggle/input/brain-stfts/stfts.npy', allow_pickle=True).item()







#%% READ EXTRA DATA
if NEEDTRAIN * (READ_EXTRA_SPEC_FILES + READ_EXTRA_EEG_FILES + READ_EXTRA_IMG_FILES + READ_EXTRA_STFT_FILES):
    train = train[['eeg_id', 'eeg_label_offset_seconds',
                   'spectrogram_id', 'spectrogram_label_offset_seconds', 
                   'patient_id', 'expert_consensus',
                   'selected', 'sign_id',
                   'seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote',
                   'seizure_vote_raw', 'lpd_vote_raw', 'gpd_vote_raw', 'lrda_vote_raw', 'grda_vote_raw', 'other_vote_raw']]
    df_extra_data = pd.read_csv('df_extra_data.csv')
    df_extra_data['selected'] = (df_extra_data.stop_time - df_extra_data.start_time).values
    df_extra_data['sign_id'] = df_extra_data['eeg_id']
    df_extra_data[['seizure_vote_raw', 'lpd_vote_raw', 'gpd_vote_raw', 'lrda_vote_raw', 'grda_vote_raw', 'other_vote_raw']] = 0
    df_extra_data['expert_consensus'] = '000'
    
    

    # df_extra_data.loc[df_extra_data.label == 'seiz', 'expert_consensus'] = 'Seizure'
    # df_extra_data.loc[df_extra_data.label == 'seiz', 'seizure_vote_raw'] = 1
    
    # df_extra_data.loc[df_extra_data.label == 'pled', 'expert_consensus'] = 'LPD'
    # df_extra_data.loc[df_extra_data.label == 'pled', 'lpd_vote_raw'] = 1
    
    # df_extra_data.loc[df_extra_data.label == 'gped', 'expert_consensus'] = 'GPD'
    # df_extra_data.loc[df_extra_data.label == 'gped', 'gpd_vote_raw'] = 1
    
    # df_extra_data.loc[df_extra_data.label == 'bckg', 'expert_consensus'] = 'Other'
    # df_extra_data.loc[df_extra_data.label == 'bckg', 'other_vote_raw'] = 1
    
    # df_extra_data.loc[df_extra_data.label == 'artf', 'expert_consensus'] = 'Other'
    # df_extra_data.loc[df_extra_data.label == 'artf', 'other_vote_raw'] = 1
    
    # df_extra_data.loc[df_extra_data.label == 'spsw', 'expert_consensus'] = 'Other'
    # df_extra_data.loc[df_extra_data.label == 'spsw', 'other_vote_raw'] = 1
    
    # df_extra_data.loc[df_extra_data.label == 'slow', 'expert_consensus'] = 'Other'
    # df_extra_data.loc[df_extra_data.label == 'slow', 'other_vote_raw'] = 1
    
    
    df_extra_data[TARGETS] = df_extra_data[['seizure_vote_raw', 'lpd_vote_raw', 'gpd_vote_raw', 'lrda_vote_raw', 'grda_vote_raw', 'other_vote_raw']].values
    
    
    df_extra_data = df_extra_data[['eeg_id', 'eeg_label_offset_seconds',
                                   'spectrogram_id', 'spectrogram_label_offset_seconds', 
                                   'patient_id', 'expert_consensus',
                                   'selected', 'sign_id',
                                   'seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote',
                                   'seizure_vote_raw', 'lpd_vote_raw', 'gpd_vote_raw', 'lrda_vote_raw', 'grda_vote_raw', 'other_vote_raw']]
    
    df_extra_data['eeg_label_offset_seconds'] = 0
    
    
    df_extra_data = df_extra_data.sort_values('selected', ascending=False).reset_index(drop=True)
    
    df_extra_data = df_extra_data.drop_duplicates('spectrogram_id').reset_index(drop=True)
    
    df_extra_data = df_extra_data[np.sum(df_extra_data[TARGETS].values, 1) > 0].reset_index(drop=True)
    
    
    
    if len(df_extra_data):
        
        if PLATFORM == 'local':
            LOAD_extra_data = './input/extra-data'
        elif PLATFORM == 'kaggle':
            LOAD_extra_data = '/kaggle/input/extra-data'
        
        
        if READ_EXTRA_SPEC_FILES and ('eeg' in DATATYPE):
            for ii in range(len(df_extra_data)):
                eegs[df_extra_data.eeg_id[ii]] = np.load(os.path.join(LOAD_extra_data, 'eeg', (df_extra_data.eeg_id[ii] + '.npy')))
        if READ_EXTRA_EEG_FILES and ('spe' in DATATYPE):
            for ii in range(len(df_extra_data)):
                spectrograms[df_extra_data.spectrogram_id[ii]] = np.load(os.path.join(LOAD_extra_data, 'spe', (df_extra_data.spectrogram_id[ii] + '.npy')))
        if READ_EXTRA_IMG_FILES and ('img' in DATATYPE):
            for ii in range(len(df_extra_data)):
                imgs[df_extra_data.eeg_id[ii]] = np.load(os.path.join(LOAD_extra_data, 'img', (df_extra_data.eeg_id[ii] + '.npy')))
        if READ_EXTRA_STFT_FILES and ('stft' in DATATYPE):
            for ii in range(len(df_extra_data)):
                stfts[df_extra_data.eeg_id[ii]] = np.load(os.path.join(LOAD_extra_data, 'stft', (df_extra_data.eeg_id[ii] + '.npy')))
    
        
    
        train = pd.concat((train, df_extra_data)).reset_index(drop=True)
        
    
    train['patient_id'] = np.array(train['patient_id'].values, dtype=str)











#%% Train DataLoader

import albumentations as albu
TARS = {'Seizure':0, 'LPD':1, 'GPD':2, 'LRDA':3, 'GRDA':4, 'Other':5}
TARS2 = {x:y for y,x in TARS.items()}

class DataGenerator(tf.keras.utils.Sequence):
    'Generates data for Keras'
    def __init__(self, data, batch_size=32, shuffle=False, augment=False, mode='train',
                 specs = None, eegs = None, imgs = None, stfts = None, targets=None): 
        
        self.targets = targets
        self.cmin = -4
        self.cmax = 6
        self.cmaps = matplotlib.colormaps['cividis'](np.linspace(0, 1, 256))[:, :3]
        
        # 0.5190
        # cividis jet Spectral gray viridis
        # magma inferno plasma viridis twilight twilight_shifted turbo Blues
        # BrBG BuGn BuPu CMRmap GnBu Greens Greys OrRd Oranges PRGn PiYG PuBu
        # PuBuGn PuOr PuRd Purples RdBu RdGy RdPu RdYlBu RdYlGn Reds Spectral
        # Wistia YlGn YlGnBu YlOrBr YlOrRd afmhot autumn binary bone brg
        # bwr cool coolwarm copper cubehelix flag gist_earth gist_gray gist_heat
        # gist_ncar gist_rainbow gist_stern gist_yarg gnuplot gnuplot2 hot hsv
        # nipy_spectral ocean pink prism rainbow seismic spring summer terrain
        # winter Accent Dark2 Paired Pastel1 Pastel2 Set1 Set2 Set3 tab10 tab20
        # tab20b tab20c grey gist_grey gist_yerg Grays

        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = False
        self.mode = mode
        self.specs = specs
        self.eegs = eegs
        self.imgs = imgs
        self.stfts = stfts
        self.on_epoch_end()
        
    def __len__(self):
        ct = int( np.ceil( len(self.data) / self.batch_size ) )
        return ct

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        x, y = self.__data_generation(indexes)
        return x, y

    def on_epoch_end(self):
        self.indexes = np.arange( len(self.data) )
        if self.shuffle: np.random.shuffle(self.indexes)
                        
    def __data_generation(self, indexes):
        if 'spe' in DATATYPE:
            x_spe = np.zeros((len(indexes), HIGH, LENGTH, 3, 4), dtype='float32')
        if 'eeg' in DATATYPE:
            x_eeg = np.zeros((len(indexes), 6, round(20 * SFREQ), 3, 4),dtype='float32')
            x_eeg2 = np.zeros((len(indexes), 4, round(50 * SFREQ), 4),dtype='float32')
        if 'img' in DATATYPE:
            x_img = np.zeros((len(indexes), IMG_HIGH, IMG_WIDE, 3, 4), dtype='float32')
        if 'stft' in DATATYPE:
            x_stft = np.zeros((len(indexes), 64, 128*4, 3, 4), dtype='float32')
        y = np.zeros((len(indexes), len(self.targets)),dtype='float32')


        for j,i in enumerate(indexes):
            row = self.data.iloc[i]
            
            
            
            row_spe = row
            row_eeg = row
            row_img = row
            row_stft = row
            
            # rows = (self.data.iloc[np.array(tf.losses.KLD(list(row[TARGETS].values), list(self.data[TARGETS].values))) <= threshold]).reset_index(drop=True)
            # rows = self.data.iloc[np.random.permutation(len(rows))].reset_index(drop=True)
            # # rows = self.data.iloc[np.random.permutation(len(self.data))].reset_index(drop=True)
            # if (self.mode=='train') and (np.random.random() > 0.5):
            #     row_spe = rows.iloc[min(0, len(rows)-1)]
            # if (self.mode=='train') and (np.random.random() > 0.5):
            #     row_eeg = rows.iloc[min(1, len(rows)-1)]
            # if (self.mode=='train') and (np.random.random() > 0.5):
            #     row_img = rows.iloc[min(2, len(rows)-1)]
            # if (self.mode=='train') and (np.random.random() > 0.5):
            #     row_stft = rows.iloc[min(3, len(rows)-1)]
                
                
                
                
                
                
            
            if self.mode=='test': 
                r_spe = 0
                r_eeg = 0
            else:
                r_spe = round( row_spe.spectrogram_label_offset_seconds / 2 )
                r_eeg = round( row_eeg.eeg_label_offset_seconds * SFREQ )
            





            if self.mode == 'train':
                x1 = np.random.rand() * LENGTH / 2
                x2 = np.random.rand() * LENGTH / 2
                x_spe_min1 = round(min(x1, x2))
                x_spe_max1 = round(max(x1, x2))
                x1 = np.random.rand() * LENGTH / 2 + LENGTH / 2
                x2 = np.random.rand() * LENGTH / 2 + LENGTH / 2
                x_spe_min2 = round(min(x1, x2))
                x_spe_max2 = round(max(x1, x2))
                


                # x1 = np.random.rand() * 20 * SFREQ / 2
                # x2 = np.random.rand() * 20 * SFREQ / 2
                # x_eeg_min11 = round(min(x1, x2))
                # x_eeg_max11 = round(max(x1, x2))
                # x1 = np.random.rand() * 20 * SFREQ / 2 + 20 * SFREQ / 2
                # x2 = np.random.rand() * 20 * SFREQ / 2 + 20 * SFREQ / 2
                # x_eeg_min21 = round(min(x1, x2))
                # x_eeg_max21 = round(max(x1, x2))
                
                # x1 = np.random.rand() * 20 * SFREQ / 2
                # x2 = np.random.rand() * 20 * SFREQ / 2
                # x_eeg_min12 = round(min(x1, x2))
                # x_eeg_max12 = round(max(x1, x2))
                # x1 = np.random.rand() * 20 * SFREQ / 2 + 20 * SFREQ / 2
                # x2 = np.random.rand() * 20 * SFREQ / 2 + 20 * SFREQ / 2
                # x_eeg_min22 = round(min(x1, x2))
                # x_eeg_max22 = round(max(x1, x2))
                
                # x1 = np.random.rand() * 20 * SFREQ / 2
                # x2 = np.random.rand() * 20 * SFREQ / 2
                # x_eeg_min13 = round(min(x1, x2))
                # x_eeg_max13 = round(max(x1, x2))
                # x1 = np.random.rand() * 20 * SFREQ / 2 + 20 * SFREQ / 2
                # x2 = np.random.rand() * 20 * SFREQ / 2 + 20 * SFREQ / 2
                # x_eeg_min23 = round(min(x1, x2))
                # x_eeg_max23 = round(max(x1, x2))
                
                
                # x1 = np.random.rand() * IMG_WIDE / 2
                # x2 = np.random.rand() * IMG_WIDE / 2
                # x_img_min1 = round(min(x1, x2))
                # x_img_max1 = round(max(x1, x2))
                # x1 = np.random.rand() * IMG_WIDE / 2 + IMG_WIDE / 2
                # x2 = np.random.rand() * IMG_WIDE / 2 + IMG_WIDE / 2
                # x_img_min2 = round(min(x1, x2))
                # x_img_max2 = round(max(x1, x2))
                




            for k in range(4):
                if 'spe' in DATATYPE: 
                    spe = self.specs[row_spe.spectrogram_id][r_spe:r_spe+300,k*100:(k+1)*100].T



                    if (spe.shape[0] != 100) or (spe.shape[1] != 300):
                        spe2 = np.zeros((100, 300))
                        spe2[:spe.shape[0], :spe.shape[1]] = spe
                        spe = spe2
                    
                    
                    spe = np.nan_to_num(spe, nan=0.0)
                    spe = np.clip(spe, np.exp(self.cmin), np.exp(self.cmax))
                    spe = np.log(spe)
                    
                    spe = np.round((spe - self.cmin) / (self.cmax - self.cmin) * 255)
                    spe = np.reshape(spe, (spe.shape[0] * spe.shape[1]))
                    spe = np.array(spe, dtype=np.int16)
                      
                    spe = self.cmaps[spe]
 
                    spe = np.reshape(spe, (100, 300, 3))
                    
                    spe = spe[:, round((spe.shape[1] - LENGTH) / 2):-round((spe.shape[1] - LENGTH) / 2), :]
                    
                    
                    
                    # if (self.mode == 'train') and (np.random.random() > 0.5): 
                    # if (self.mode == 'train'):
                    #     spe[:, x_spe_min1:x_spe_max1, :] = 0
                    #     spe[:, x_spe_min2:x_spe_max2, :] = 0
                    
                    
                    
                    if (self.mode == 'train') and (np.random.random() > 0.5):
                        spe[:, x_spe_min1:x_spe_max1, :] = 0
                    if (self.mode == 'train') and (np.random.random() > 0.5):
                        spe[:, x_spe_min2:x_spe_max2, :] = 0
                    
    
    
    
    
                    x_spe[j, round((HIGH - spe.shape[0]) / 2):round((HIGH + spe.shape[0]) / 2), :, :, k] = spe
                    x_spe[j, :, :, 0, k] = (x_spe[j, :, :, 0, k] - 0.485) / (0.229**2)
                    x_spe[j, :, :, 1, k] = (x_spe[j, :, :, 1, k] - 0.456) / (0.224**2)
                    x_spe[j, :, :, 2, k] = (x_spe[j, :, :, 2, k] - 0.406) / (0.225**2)





                if 'eeg' in DATATYPE:
                    eeg = self.eegs[row_eeg.eeg_id][:, r_eeg:r_eeg + round(50 * SFREQ), k]




                    if eeg.shape[1] < 50 * SFREQ:
                        eeg = np.concatenate((eeg, eeg), 1)
                        eeg = eeg[:, :50 * SFREQ]
                        
                        
                    # eeg1 = eeg[:, round(10*SFREQ) : round(30*SFREQ)]
                    
                    # eeg2 = eeg[:, round(15*SFREQ) : round(35*SFREQ)]
                    
                    # eeg3 = eeg[:, round(20*SFREQ) : round(40*SFREQ)]
                    
                    eeg1 = eeg[:, round(10*SFREQ) : round(30*SFREQ)]
                    
                    eeg2 = eeg[:, round(15*SFREQ) : round(35*SFREQ)]
                    
                    eeg3 = eeg[:, round(20*SFREQ) : round(40*SFREQ)]
                    

                    
                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     eeg1[:, x_eeg_min11:x_eeg_max11] = 0
                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     eeg1[:, x_eeg_min21:x_eeg_max21] = 0
                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     eeg2[:, x_eeg_min12:x_eeg_max12] = 0
                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     eeg2[:, x_eeg_min22:x_eeg_max22] = 0
                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     eeg3[:, x_eeg_min13:x_eeg_max13] = 0
                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     eeg3[:, x_eeg_min23:x_eeg_max23] = 0
                        
                    

                    x_eeg[j, 1:5, :, 0, k] = eeg1
                    x_eeg[j, 1:5, :, 1, k] = eeg2
                    x_eeg[j, 1:5, :, 2, k] = eeg3
                    x_eeg[j, :, :, :, k] = (x_eeg[j, :, :, :, k] - np.mean(x_eeg[j, :, :, :, k], 1, keepdims=True)) / (np.std(x_eeg[j, :, :, :, k], 1, keepdims=True) + 1e-6)
                    
                    
                    eeg = (eeg - np.mean(eeg, 1, keepdims=True)) / (1e-6 + np.std(eeg, 1, keepdims=True))
                    x_eeg2[j, :, :, k] = eeg
                    


                if 'img' in DATATYPE:
                    if self.mode == 'test':
                        img = self.imgs[row_img.eeg_id][:, :, k, :]
                    else:
                        img = self.imgs[row_img.sign_id][:, :, k, :]

                    


                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     img[:, x_img_min1:x_img_max1, :] = 0
                    # if (self.mode == 'train') and (np.random.random() > 0.5):
                    #     img[:, x_img_min2:x_img_max2, :] = 0
                   


                    
                    x_img[j, :, :, :, k] = img

 
                    
 
    
 
                if 'stft' in DATATYPE:
                    if self.mode == 'test':
                        stft = self.stfts[row_stft.eeg_id][:, :, :, k]
                    else:
                        stft = self.stfts[row_stft.sign_id][:, :, :, k]
                    
                    
                    
                    if (stft.shape[1] != 64) or (stft.shape[2] != 256):
                        stft2 = np.zeros((4, 64, 128))
                        stft2[:, :stft.shape[1], :stft.shape[2]] = stft
                        stft = stft2 



                    stft = np.concatenate([stft[0, :, :], stft[1, :, :],
                                           stft[2, :, :], stft[3, :, :]], 1)
                    
                    
                    stft = np.clip(stft, np.exp(self.cmin), np.exp(self.cmax))
                    stft = np.log(stft)
                    stft = np.nan_to_num(stft, nan=0.0)
                    
                    
                    
                    
                    stft = np.round((stft - self.cmin) / (self.cmax - self.cmin) * 255)
                    stft = np.reshape(stft, (stft.shape[0] * stft.shape[1]))
                    stft = np.array(stft, dtype=np.int16)
                    
                    
                    stft = self.cmaps[stft]
                    
                    
                    stft = np.reshape(stft, (64, 128*4, 3))
                    
                    
                    
                    
                    
                    
                    
                    
                    
                    x_stft[j, :, :, :, k] = stft
                    x_stft[j, :, :, 0, k] = (x_stft[j, :, :, 0, k] - 0.485) / (0.229**2)
                    x_stft[j, :, :, 1, k] = (x_stft[j, :, :, 1, k] - 0.456) / (0.224**2)
                    x_stft[j, :, :, 2, k] = (x_stft[j, :, :, 2, k] - 0.406) / (0.225**2)
                    



            if self.mode!='test':
                label = 0
                if 'spe' in DATATYPE:
                    label = label + row_spe[self.targets].values
                if 'eeg' in DATATYPE:
                    label = label + row_spe[self.targets].values
                if 'img' in DATATYPE:
                    label = label + row_img[self.targets].values
                if 'stft' in DATATYPE:
                    label = label + row_stft[self.targets].values
                label = label / len(DATATYPE)
                
                if self.mode=='train' and sum(label==1):
                    xx = (np.random.random() + 1) * 0.005
                    label[label==0] = xx
                    label[label==1] = 1 - 5 * xx
                y[j] = label
        

        




        # if self.mode == 'train':
        #     if np.random.randn() > 0:
        #         x_spe = x_spe[:, ::-1, :, :, :]
        #     if np.random.randn() > 0:
        #         x_eeg = x_eeg[:, ::-1, :, :, :]
        #     if np.random.randn() > 0:
        #         x_img = x_eeg[:, ::-1, :, :, :]
        #     if np.random.randn() > 0:
        #         x_stft = x_stft[:, ::-1, :, :, :]
        
        
        x = list()
        if 'spe' in DATATYPE:
            x.append(x_spe)
            
            
        if 'eeg' in DATATYPE:
            x.append(x_eeg)
            x.append(x_eeg2)
            
            
        if 'img' in DATATYPE:
            if self.mode == 'train':
                aug_img = (np.random.random((x_img.shape[0], 1, 1, 1, 1)) > 0.5) * 2 - 1
                x_img = x_img * aug_img
            x.append(x_img)
        
        
        if 'stft' in DATATYPE:
            x.append(x_stft)


        return x, y






#%% Train Scheduler
if NEEDTRAIN:
    LR_MAX1 = 1e-3
    EPOCHS1 = 6
    def lrfn1(epoch):
        xx = [LR_MAX1, LR_MAX1, LR_MAX1]
        if epoch < len(xx):
            return xx[epoch]
        else:
            return 1e-4
    LR1 = tf.keras.callbacks.LearningRateScheduler(lrfn1, verbose = True)
    
    
    
    LR_MAX2 = 1e-3
    EPOCHS2 = 4
    def lrfn2(epoch):
        xx = [LR_MAX2]
        if epoch < len(xx):
            return xx[epoch]
        else:
            return 1e-4
    LR2 = tf.keras.callbacks.LearningRateScheduler(lrfn2, verbose = True)
    
    
    
    LR_MAX3 = 1e-4
    EPOCHS3 = 6
    def lrfn3(epoch):
        xx = [LR_MAX3, LR_MAX3, LR_MAX3]
        if epoch < len(xx):
            return xx[epoch]
        else:
            return 1e-5
    LR3 = tf.keras.callbacks.LearningRateScheduler(lrfn3, verbose = True)
    




#%% Build EfficientNetB2 Model
import efficientnet.tfkeras as efn

def build_model(TARGETS_PRETRAIN):
    inp = list()
    if 'spe' in DATATYPE:
        inp_spe = tf.keras.Input(shape=(HIGH, LENGTH, 3, 4))
        x_spe1 = inp_spe[:,:,:,:,0]
        x_spe2 = inp_spe[:,:,:,:,1]
        x_spe3 = inp_spe[:,:,:,:,2]
        x_spe4 = inp_spe[:,:,:,:,3]
        x_spe = tf.keras.layers.Concatenate(axis=1)([x_spe1, x_spe2, x_spe3, x_spe4])
        
        # base_model_spe = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False, include_preprocessing=False, weights=None)
        # base_model_spe._name = 'spe_extractor'
        # if NEEDTRAIN:
        #     if PLATFORM == 'local':
        #         base_model_spe.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnetv2-b0_notop.h5')
        #     if PLATFORM == 'kaggle':
        #         base_model_spe.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnetv2-b0_notop.h5')
        base_model_spe = efn.EfficientNetB0(include_top=False, weights=None, input_shape=None)
        base_model_spe._name = 'spe_extractor'
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_spe.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_spe.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
        
        x_spe = base_model_spe(x_spe)
        x_spe = tf.keras.layers.GlobalAveragePooling2D()(x_spe)
        x_spe = tf.nn.l2_normalize(x_spe, -1)
        
        inp.append(inp_spe)
        
        y = x_spe
        



    if 'eeg' in DATATYPE:
        inp_eeg = tf.keras.Input(shape=(6, round(20 * SFREQ), 3, 4))
        x_eeg1 = inp_eeg[:, :, :, :, 0]
        x_eeg2 = inp_eeg[:, :, :, :, 1]
        x_eeg3 = inp_eeg[:, :, :, :, 2]
        x_eeg4 = inp_eeg[:, :, :, :, 3]
        
        x_eeg = tf.keras.layers.Concatenate(axis=1)([x_eeg1, x_eeg2, x_eeg3, x_eeg4])
        # x_eeg = tf.keras.layers.Concatenate(axis=3)([x_eeg, x_eeg, x_eeg])
        # x_eeg11 = tf.keras.layers.Concatenate(axis=1)([x_eeg1, x_eeg2 * 0, x_eeg3, x_eeg4 * 0])
        # x_eeg22 = tf.keras.layers.Concatenate(axis=1)([x_eeg1 * 0, x_eeg2, x_eeg3 * 0, x_eeg4])
        # x_eeg33 = tf.keras.layers.Concatenate(axis=1)([x_eeg1, x_eeg2, x_eeg3, x_eeg4]) * 0
        # x_eeg = tf.keras.layers.Concatenate(axis=3)([x_eeg11, x_eeg22, x_eeg33])
        
        # base_model_eeg = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False, include_preprocessing=False, weights=None)
        # base_model_eeg._name = 'eeg_extractor'
        # if NEEDTRAIN:
        #     if PLATFORM == 'local':
        #         base_model_eeg.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnetv2-b0_notop.h5')
        #     if PLATFORM == 'kaggle':
        #         base_model_eeg.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnetv2-b0_notop.h5')
        base_model_eeg = efn.EfficientNetB0(include_top=False, weights=None, input_shape=None)
        base_model_eeg._name = 'eeg_extractor'
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_eeg.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_eeg.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
        
        x_eeg = base_model_eeg(x_eeg)
        x_eeg = tf.keras.layers.GlobalAveragePooling2D()(x_eeg)[:, :round(x_eeg.shape[1] / 2)]
        x_eeg = tf.nn.l2_normalize(x_eeg, -1) * 0.707
        
        inp.append(inp_eeg)
        
        if 'spe' in DATATYPE:
            y = tf.keras.layers.Concatenate(axis=1)([y, x_eeg])
        else:
            y = x_eeg
        
    
    if 'eeg' in DATATYPE:
        inp_eeg2 = tf.keras.Input(shape=(4, round(50 * SFREQ), 4))
        
        
        kernal_num = 64
        x_eeg1 = inp_eeg2[:, :, :, 0:1]
        x_eeg2 = inp_eeg2[:, :, :, 1:2]
        x_eeg3 = inp_eeg2[:, :, :, 2:3]
        x_eeg4 = inp_eeg2[:, :, :, 3:4]
        x_eeg = tf.keras.layers.Concatenate(axis=1)([x_eeg1, x_eeg2, x_eeg3, x_eeg4])
        

        x_eeg = tf.keras.layers.Conv2D(kernal_num*1, (1, 8), padding='valid')(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        x_eeg = tf.keras.layers.Conv2D(kernal_num*1, (1, 6), padding='valid')(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        x_eeg = tf.keras.layers.Conv2D(kernal_num*1, (1, 4), padding='valid')(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        
        
        
        
        
        
        x_eeg = tf.keras.layers.Conv2D(kernal_num*2, (4, 4), padding='valid', strides=(4, 1))(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        x_eeg = tf.keras.layers.Conv2D(kernal_num*4, (1, 8), padding='valid')(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        x_eeg = tf.keras.layers.Conv2D(kernal_num*4, (1, 4), padding='valid')(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        
        
        
        
        
        
        
        x_eeg = tf.keras.layers.Conv2D(kernal_num*8, (4, 4), padding='valid', strides=(4, 1))(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        x_eeg = tf.keras.layers.Conv2D(kernal_num*16, (1, 4), padding='valid')(x_eeg)
        x_eeg = tf.keras.layers.Activation('ReLU')(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)
        
        
        x_eeg = tf.keras.layers.GlobalAveragePooling2D()(x_eeg)[:, :round(x_eeg.shape[1] / 2)]

        x_eeg = tf.nn.l2_normalize(x_eeg, -1) * 707

        inp.append(inp_eeg2)
        

        y = tf.keras.layers.Concatenate(axis=1)([y, x_eeg])
        
        
        
        
    if 'img' in DATATYPE:
        inp_img = tf.keras.Input(shape=(IMG_HIGH, IMG_WIDE, 3, 4))
        x_img1 = inp_img[:, :, :, :, 0]
        x_img2 = inp_img[:, :, :, :, 1]
        x_img3 = inp_img[:, :, :, :, 2]
        x_img4 = inp_img[:, :, :, :, 3]
        # x_img = tf.keras.layers.Concatenate(axis=1)([x_img1, x_img2, x_img3, x_img4])
        x_img = tf.keras.layers.Concatenate(axis=1)([x_img1, x_img3, x_img4, x_img2])
        # x_img = tf.keras.layers.Concatenate(axis=3)([x_img, x_img, x_img])
        
        # base_model_img = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False, include_preprocessing=False, weights=None)
        # base_model_img._name = 'img_extractor'
        # if NEEDTRAIN:
        #     if PLATFORM == 'local':
        #         base_model_img.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnetv2-b0_notop.h5')
        #     if PLATFORM == 'kaggle':
        #         base_model_img.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnetv2-b0_notop.h5')
        base_model_img = efn.EfficientNetB0(include_top=False, weights=None, input_shape=None)
        base_model_img._name = 'img_extractor'
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_img.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_img.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
        
        x_img = base_model_img(x_img)
        x_img = tf.keras.layers.GlobalAveragePooling2D()(x_img)
        x_img = tf.nn.l2_normalize(x_img, -1)
        
        inp.append(inp_img)
        
        if ('spe' in DATATYPE) or ('eeg' in DATATYPE):
            y = tf.keras.layers.Concatenate(axis=1)([y, x_img])
        else:
            y = x_img
    
    
    
    if 'stft' in DATATYPE:
        inp_stft = tf.keras.Input(shape=(64, 128*4, 3, 4))
        x_stft1 = inp_stft[:,:,:,:,0]
        x_stft2 = inp_stft[:,:,:,:,1]
        x_stft3 = inp_stft[:,:,:,:,2]
        x_stft4 = inp_stft[:,:,:,:,3]
        x_stft = tf.keras.layers.Concatenate(axis=1)([x_stft1, x_stft2, x_stft3, x_stft4])
        
        base_model_stft = efn.EfficientNetB0(include_top=False, weights=None, input_shape=None)
        base_model_stft._name = 'stft_extractor'
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_stft.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_stft.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnet-b0_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
        
        x_stft = base_model_stft(x_stft)
        x_stft = tf.keras.layers.GlobalAveragePooling2D()(x_stft)
        x_stft = tf.nn.l2_normalize(x_stft, -1)
        
        inp.append(inp_stft)
        
        if ('spe' in DATATYPE) or ('eeg' in DATATYPE) or ('img' in DATATYPE):
            y = tf.keras.layers.Concatenate(axis=1)([y, x_stft])
        else:
            y = x_stft
    
 
    
    y = tf.keras.layers.Dense(len(TARGETS_PRETRAIN), activation='softmax', dtype='float32')(y)


    model = tf.keras.Model(inputs=inp, outputs=y)
    
        
    return model









#%% Train Model
def my_loss(y_true, y_pred):
    loss1 = tf.keras.losses.KLD(y_true, y_pred)
    loss2 = tf.keras.losses.categorical_focal_crossentropy(y_true, y_pred)
    return loss1 + loss2



if NEEDTRAIN:
    from sklearn.model_selection import GroupKFold
    import tensorflow.keras.backend as K, gc
    import itertools
    
    gkf = GroupKFold(n_splits=NSPLIT)
    
    train['pd_rda_vote'] = train.lpd_vote + train.gpd_vote + train.lrda_vote + train.grda_vote
    
    
    
    
    # xx = (train[TARGETS_RAW].sum(1)==1) * (train.other_vote==1)
    # train = train.copy()
    # train.lrda_vote[xx] = 0.25
    # train.grda_vote[xx] = 0.25
    # train.other_vote[xx] = 0.5
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    # for i, (train_index, valid_index) in enumerate(gkf.split(train, train.expert_consensus, train.patient_id)):  
    for i, (train_index, valid_index) in enumerate(gkf.split(train, train[TARGETS_RAW].sum(1), train.patient_id)):  
        if (i+1) >= 1:
            print(valid_index)
            print('#'*25)
            print(f'### Fold {i+1}')
            
            
            TARGETS_PRETRAIN = ['seizure_vote', 'pd_rda_vote', 'other_vote']
            
            
            df_train_stage1 = train.iloc[train_index].reset_index(drop=True)
            df_valid_stage1 = train.iloc[valid_index].reset_index(drop=True)
            

            if 1 in STAGETRAIN:
                df_train_stage1 = df_train_stage1.sort_values('selected', ascending=False).reset_index(drop=True).drop_duplicates('eeg_id').reset_index(drop=True)
                df_valid_stage1 = df_valid_stage1.sort_values('selected', ascending=False).reset_index(drop=True).drop_duplicates('eeg_id').reset_index(drop=True)
                
                df_train_stage2 = df_train_stage1[df_train_stage1[TARGETS_RAW].sum(1) > 1].reset_index(drop=True)
                df_valid_stage2 = df_valid_stage1[df_valid_stage1[TARGETS_RAW].sum(1) > 1].reset_index(drop=True)
            else:
                df_train_stage2 = df_train_stage1.sort_values('selected', ascending=False).reset_index(drop=True).drop_duplicates('eeg_id').reset_index(drop=True)
                df_valid_stage2 = df_valid_stage1.sort_values('selected', ascending=False).reset_index(drop=True).drop_duplicates('eeg_id').reset_index(drop=True)
            
            
            
            xx = np.sum(df_train_stage2[TARGETS_RAW].values, 1) >= 10
            df_train_stage3 = df_train_stage2[xx].reset_index(drop=True)
            xx = np.sum(df_valid_stage2[TARGETS_RAW].values, 1) >= 10
            df_valid_stage3 = df_valid_stage2[xx].reset_index(drop=True)
            
            
            df_train_stage3 = df_train_stage3.sort_values('selected', ascending=False).reset_index(drop=True).drop_duplicates('spectrogram_id').reset_index(drop=True)
            df_valid_stage3 = df_valid_stage3.sort_values('selected', ascending=False).reset_index(drop=True).drop_duplicates('spectrogram_id').reset_index(drop=True)





            if 1 in STAGETRAIN:
                print(f'stage1 ### train size {len(df_train_stage1)}, valid size {len(df_valid_stage1)}')
                print('#'*25)
                train_gen = DataGenerator(df_train_stage1, shuffle=True, batch_size=BATCHSIZE, specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS)
                valid_gen = DataGenerator(df_valid_stage1, shuffle=False, batch_size=BATCHSIZE*2, mode='valid', specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS)

                LR = LR1
                EPOCHS = EPOCHS1  
                LR_MAX = LR_MAX1
                
                callbacks_list = [LR,
                                  tf.keras.callbacks.ModelCheckpoint(filepath=f'f{i}_stage1.h5',
                                                                      monitor='val_loss', mode='min',
                                                                      save_weights_only=True, save_best_only=True),
                                  tf.keras.callbacks.EarlyStopping(patience=5, monitor='val_loss', mode='min')]
                with strategy.scope():
                    model = build_model(TARGETS)
                    opt = tf.keras.optimizers.Adam(learning_rate = LR_MAX, clipvalue=1.0)
                    # loss = tf.keras.losses.KLDivergence()
                    loss = my_loss
                    model.compile(loss=loss, optimizer = opt)
                    
                history = model.fit(train_gen, verbose=1, validation_data = valid_gen,
                                    epochs=EPOCHS, callbacks = callbacks_list)

                loss = history.history['loss']
                val_loss = history.history['val_loss']
                epochs = range(1, len(loss) + 1)
                plt.plot(epochs, loss, 'bo', label='loss')
                plt.plot(epochs, val_loss, 'b', label='val_loss')
                plt.title(f'loss: {round(min(loss)*10000)/10000}, val loss: {round(min(val_loss)*10000)/10000}', fontsize=12)
                plt.legend()
                plt.savefig(f'f{i}_stage1.pdf')
                plt.close()


                model.load_weights(f'f{i}_stage1.h5')
                valid_stage1 = df_valid_stage1[TARGETS].values
                pre_stage1 = model.predict(valid_gen)
                # pre_stage2[:, 5] = np.sum(pre_stage2[:, 5:], 1)
                # pre_stage2 = pre_stage2[:, :6]
                
                KLD = str(np.mean(tf.losses.KLD(valid_stage1, pre_stage1)))[:6]
                
                
                cm = confusion_matrix(np.argmax(valid_stage1, 1), np.argmax(pre_stage1, 1))
                cm = cm / np.sum(cm, 1, keepdims=True)
                
                plt.figure()
                plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
                plt.title('Confusion Matrix')
                plt.colorbar()
                tick_marks = np.arange(6)
                plt.xticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
                plt.yticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
                thresh = cm.max() / 2.
                for ii, jj in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
                    plt.text(jj, ii, str(round(cm[ii, jj] * 1e4) * 1e-2)[:5], horizontalalignment="center", color="white" if cm[ii, jj] > thresh else "black", fontsize=10)
                plt.xlabel('Predicted label')
                plt.ylabel('True label')
                plt.tight_layout()
                plt.savefig(f'f{i}_stage1_cm_{KLD}.pdf')
                plt.close()


                del model, history, train_gen, valid_gen
                K.clear_session()
                reset_default_graph()
                gc.collect()
            
            

            if 2 in STAGETRAIN:
                print(f'stage2 ### train size {len(df_train_stage2)}, valid size {len(df_valid_stage2)}')
                print('#'*25)
                train_gen = DataGenerator(df_train_stage2, shuffle=True, batch_size=BATCHSIZE, specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS)
                valid_gen = DataGenerator(df_valid_stage2, shuffle=False, batch_size=BATCHSIZE*2, mode='valid', specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS)
                
                if 1 in STAGETRAIN:
                    LR = LR2
                    EPOCHS = EPOCHS2
                    LR_MAX = LR_MAX2
                else:
                    LR = LR1
                    EPOCHS = EPOCHS1  
                    LR_MAX = LR_MAX1
                    
                
                callbacks_list = [LR,
                                  tf.keras.callbacks.ModelCheckpoint(filepath=f'f{i}_stage2.h5',
                                                                      monitor='val_loss', mode='min',
                                                                      save_weights_only=True, save_best_only=True),
                                  tf.keras.callbacks.EarlyStopping(patience=5, monitor='val_loss', mode='min')]
                with strategy.scope():
                    model = build_model(TARGETS)
                    
                    # opt = tf.keras.optimizers.Adam(learning_rate = LR_MAX, clipnorm=1.0) # fold2-0.4687
                    # opt = tf.keras.optimizers.Adam(learning_rate = LR_MAX, clipvalue=0.1) # fold2-0.4753
                    # opt = tf.keras.optimizers.Adam(learning_rate = LR_MAX, clipvalue=5.0) # fold2-0.4611
                    opt = tf.keras.optimizers.Adam(learning_rate = LR_MAX, clipvalue=1.0) # fold2-0.4611
                    
                    # loss = tf.keras.losses.KLDivergence()
                    loss = my_loss
                    model.compile(loss=loss, optimizer = opt)
                
                if 1 in STAGETRAIN:
                    model.load_weights(f'f{i}_stage1.h5')
                    
                history = model.fit(train_gen, verbose=1, validation_data = valid_gen,
                                    epochs=EPOCHS, callbacks = callbacks_list)

                loss = history.history['loss']
                val_loss = history.history['val_loss']
                epochs = range(1, len(loss) + 1)
                plt.plot(epochs, loss, 'bo', label='loss')
                plt.plot(epochs, val_loss, 'b', label='val_loss')
                plt.title(f'loss: {round(min(loss)*10000)/10000}, val loss: {round(min(val_loss)*10000)/10000}', fontsize=12)
                plt.legend()
                plt.savefig(f'f{i}_stage2.pdf')
                plt.close()


                model.load_weights(f'f{i}_stage2.h5')
                valid_stage2 = df_valid_stage2[TARGETS].values
                pre_stage2 = model.predict(valid_gen)
                # pre_stage2[:, 5] = np.sum(pre_stage2[:, 5:], 1)
                # pre_stage2 = pre_stage2[:, :6]
                
                KLD = str(np.mean(tf.losses.KLD(valid_stage2, pre_stage2)))[:6]
                
                
                cm = confusion_matrix(np.argmax(valid_stage2, 1), np.argmax(pre_stage2, 1))
                cm = cm / np.sum(cm, 1, keepdims=True)
                
                plt.figure()
                plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
                plt.title('Confusion Matrix')
                plt.colorbar()
                tick_marks = np.arange(6)
                plt.xticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
                plt.yticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
                thresh = cm.max() / 2.
                for ii, jj in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
                    plt.text(jj, ii, str(round(cm[ii, jj] * 1e4) * 1e-2)[:5], horizontalalignment="center", color="white" if cm[ii, jj] > thresh else "black", fontsize=10)
                plt.xlabel('Predicted label')
                plt.ylabel('True label')
                plt.tight_layout()
                plt.savefig(f'f{i}_stage2_cm_{KLD}.pdf')
                plt.close()


                del model, history, train_gen, valid_gen
                K.clear_session()
                reset_default_graph()
                gc.collect()



            if 3 in STAGETRAIN:
                print(f'stage3 ### train size {len(df_train_stage3)}, valid size {len(df_valid_stage3)}')
                print('#'*25)
                train_gen = DataGenerator(df_train_stage3, shuffle=True, batch_size=BATCHSIZE, specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS)
                valid_gen = DataGenerator(df_valid_stage3, shuffle=False, batch_size=BATCHSIZE*2, mode='valid', specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS)
                
                LR = LR3
                EPOCHS = EPOCHS3
                LR_MAX = LR_MAX3
                
                callbacks_list = [LR,
                                  tf.keras.callbacks.ModelCheckpoint(filepath=f'f{i}_stage3.h5',
                                                                      monitor='val_loss', mode='min',
                                                                      save_weights_only=True, save_best_only=True),
                                  tf.keras.callbacks.EarlyStopping(patience=5, monitor='val_loss', mode='min')]
                with strategy.scope():
                    model = build_model(TARGETS)
                    opt = tf.keras.optimizers.Adam(learning_rate = LR_MAX, clipvalue=1.0)
                    loss = tf.keras.losses.KLDivergence()
                    # loss = my_loss
                    model.compile(loss=loss, optimizer = opt)
                    

                model.load_weights(f'f{i}_stage2.h5')
                    
                history = model.fit(train_gen, verbose=1, validation_data = valid_gen,
                                    epochs=EPOCHS, callbacks = callbacks_list)

                loss = history.history['loss']
                val_loss = history.history['val_loss']
                epochs = range(1, len(loss) + 1)
                plt.plot(epochs, loss, 'bo', label='loss')
                plt.plot(epochs, val_loss, 'b', label='val_loss')
                plt.title(f'loss: {round(min(loss)*10000)/10000}, val loss: {round(min(val_loss)*10000)/10000}', fontsize=12)
                plt.legend()
                plt.savefig(f'f{i}_stage3.pdf')
                plt.close()
                
                model.load_weights(f'f{i}_stage3.h5')
                valid_stage3 = df_valid_stage3[TARGETS].values
                pre_stage3 = model.predict(valid_gen)
                cm = confusion_matrix(np.argmax(valid_stage3, 1), np.argmax(pre_stage3, 1))
                cm = cm / np.sum(cm, 1, keepdims=True)
                
                plt.figure()
                plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
                plt.title('Confusion Matrix')
                plt.colorbar()
                tick_marks = np.arange(6)
                plt.xticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
                plt.yticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
                thresh = cm.max() / 2.
                for ii, jj in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
                    plt.text(jj, ii, str(round(cm[ii, jj] * 1e4) * 1e-2)[:5], horizontalalignment="center", color="white" if cm[ii, jj] > thresh else "black", fontsize=10)
                plt.xlabel('Predicted label')
                plt.ylabel('True label')
                plt.tight_layout()
                plt.savefig(f'f{i}_stage3_cm.pdf')
                plt.close()


                del model, history, train_gen, valid_gen
                K.clear_session()
                reset_default_graph()
                gc.collect()

                
                
                
            # if 1 in STAGETRAIN:
            #     model1 = build_model(TARGETS_PRETRAIN)
            #     model1.load_weights(f'f{i}_stage1.h5')
            #     valid_gen1 = DataGenerator(df_valid_stage3, shuffle=False, batch_size=BATCHSIZE*2, mode='valid', specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS_PRETRAIN)
            #     pre_stage1 = model1.predict(valid_gen1)
            #     del model1
                
                
            #     pre_stage13 = pre_stage3 * 1
            #     pre_stage13[:, 0:1] = pre_stage13[:, 0:1] * pre_stage1[:, 0:1]
            #     pre_stage13[:, 1:5] = pre_stage13[:, 1:5] * pre_stage1[:, 1:2]
            #     pre_stage13[:, 5:6] = pre_stage13[:, 5:6] * pre_stage1[:, 2:3]
            #     pre_stage13 = np.sqrt(pre_stage13)
            #     pre_stage13 = pre_stage13 / np.sum(pre_stage13, 1, keepdims=True)
                
            #     kl13 = tf.losses.KLD(valid_stage3, pre_stage13)
            #     kl13 = str(np.mean(kl13))[:6]
                
            #     cm = confusion_matrix(np.argmax(valid_stage3, 1), np.argmax(pre_stage13, 1))
            #     cm = cm / np.sum(cm, 1, keepdims=True)
                
            #     plt.figure()
            #     plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            #     plt.title('Confusion Matrix')
            #     plt.colorbar()
            #     tick_marks = np.arange(6)
            #     plt.xticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
            #     plt.yticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
            #     thresh = cm.max() / 2.
            #     for ii, jj in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            #         plt.text(jj, ii, str(round(cm[ii, jj] * 1e4) * 1e-2)[:5], horizontalalignment="center", color="white" if cm[ii, jj] > thresh else "black", fontsize=10)
            #     plt.xlabel('Predicted label')
            #     plt.ylabel('True label')
            #     plt.tight_layout()
            #     plt.savefig(f'f{i}_stage13_cm_{kl13}.pdf')
            #     plt.close()
                
                
                
            # if 2 in STAGETRAIN:
            #     model2 = build_model(TARGETS)
            #     model2.load_weights(f'f{i}_stage2.h5')
            #     valid_gen2 = DataGenerator(df_valid_stage3, shuffle=False, batch_size=BATCHSIZE*2, mode='valid', specs = spectrograms, eegs = eegs, imgs=imgs, stfts=stfts, targets=TARGETS)
            #     pre_stage2 = model2.predict(valid_gen2)
            #     del model2

            #     pre_stage23 = pre_stage3 * 1
            #     pre_stage23[np.argmax(pre_stage2, 1) == 0] = pre_stage2[np.argmax(pre_stage2, 1) == 0
            #                                                             ] * 1 + pre_stage3[np.argmax(pre_stage2, 1) == 0] * 0
                
                
            #     kl23 = tf.losses.KLD(valid_stage3, pre_stage23)
            #     kl23 = str(np.mean(kl23))[:6]
                
                
            #     cm = confusion_matrix(np.argmax(valid_stage3, 1), np.argmax(pre_stage23, 1))
            #     cm = cm / np.sum(cm, 1, keepdims=True)
                
            #     plt.figure()
            #     plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            #     plt.title('Confusion Matrix')
            #     plt.colorbar()
            #     tick_marks = np.arange(6)
            #     plt.xticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
            #     plt.yticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
            #     thresh = cm.max() / 2.
            #     for ii, jj in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            #         plt.text(jj, ii, str(round(cm[ii, jj] * 1e4) * 1e-2)[:5], horizontalalignment="center", color="white" if cm[ii, jj] > thresh else "black", fontsize=10)
            #     plt.xlabel('Predicted label')
            #     plt.ylabel('True label')
            #     plt.tight_layout()
            #     plt.savefig(f'f{i}_stage23_cm_{kl23}.pdf')
            #     plt.close()
                
            # K.clear_session()
            # reset_default_graph()
            # gc.collect()

                
                
                
                
                
                
                
             






#%% Infer Test and Create Submission CSV
if not NEEDTRAIN: 
    if PLATFORM == 'local':
        test = pd.read_csv('./input/hms-harmful-brain-activity-classification/test.csv')
    elif PLATFORM == 'kaggle':
        test = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/test.csv')
    print('Test shape',test.shape)
    test.head()
    
    
    if 'spe' in DATATYPE:
        
        
        # READ ALL SPECTROGRAMS
        if PLATFORM == 'local':
            PATH2 = './input/hms-harmful-brain-activity-classification/test_spectrograms/'
        elif PLATFORM == 'kaggle':
            PATH2 = '/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/'
        
        files2 = os.listdir(PATH2)
        print(f'There are {len(files2)} test spectrogram parquets')
            
        spectrograms2 = {}
        for i,f in enumerate(files2):
            if i%100==0: print(i,', ',end='')
            tmp = pd.read_parquet(f'{PATH2}{f}')
            name = int(f.split('.')[0])
            spectrograms2[name] = tmp.iloc[:,1:].values
        
    # RENAME FOR DATALOADER
    # test = test.rename({'spectrogram_id':'spec_id'},axis=1)
    
    
    #%% READ ALL EEGS
    from scipy import signal
    
    # READ ALL SPECTROGRAMS
    if PLATFORM == 'local':
        PATH2 = './input/hms-harmful-brain-activity-classification/test_eegs/'
    elif PLATFORM == 'kaggle':
        PATH2 = '/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/'
        
    files2 = os.listdir(PATH2)
    print(f'There are {len(files2)} test eeg parquets')
      
    eegs2 = {}
    imgs2 = {}
    stfts2 = {}
    b, a = signal.butter(3, np.float32(filter_range)*2/SFREQ, 'bandpass')
    for i,f in enumerate(files2):
        if i%100==0: print(i,', ',end='')
        eeg_default = pd.read_parquet(f'{PATH2}{f}')
        name = int(f.split('.')[0])
        
        if len(test[test.eeg_id == name]) > 0:
            # time_temp = 0
            # time_start =  round(time_temp * 200 + (50 - EEG_LENGTH) / 2 * 200) 
            # time_stop =  round(time_temp * 200 + (50 + EEG_LENGTH) / 2 * 200)
    
            # eeg_default = raw_eeg.loc[time_start: (time_stop - 1), :].reset_index(drop=True)
            
            list_eeg = list()
            list_img = list()
            list_stft = list()
            for region in BRAIN.keys():
                eeg = np.zeros((len(BRAIN[region]), eeg_default.shape[0]), dtype=np.float32)
                for chan_i, chan in enumerate(BRAIN[region]):
                    eeg[chan_i, :] = (eeg_default.loc[:, chan.split('-')[0]] - eeg_default.loc[:, chan.split('-')[1]]).values
                
                eeg[np.isnan(eeg)] = 0
                
                if 200 != SFREQ:
                    eeg = signal.resample_poly(eeg, SFREQ, 200, axis=1)
    
                eeg = signal.filtfilt(b, a, eeg, axis=1)
                
                
                time_temp  = 0
                time_start = round(time_temp * SFREQ + (50 - EEG_LENGTH) / 2 * SFREQ) 
                time_stop  = round(time_temp * SFREQ + (50 + EEG_LENGTH) / 2 * SFREQ)
                
                
                
                list_img.append(eeg[:, time_start:time_stop])
                
                if 'stft' in DATATYPE:
                    frequencies, times, Sxx = signal.spectrogram(eeg[:, round(time_temp*SFREQ):round((time_temp+50)*SFREQ)],
                                                                SFREQ, nperseg=256, noverlap=219,
                                                                nfft=320)
                    valid_freq = (frequencies > 0.0) & (frequencies <= 20)
                    Sxx_filtered = Sxx[:, valid_freq, :-1]

                    Sxx_filtered = np.reshape(Sxx_filtered, (Sxx_filtered.shape[0],
                                                             Sxx_filtered.shape[1],
                                                             Sxx_filtered.shape[2],
                                                             1))
        
                    list_stft.append(Sxx_filtered)

                list_eeg.append(np.reshape(eeg, (eeg.shape[0], eeg.shape[1], 1)))



            list_eeg = np.concatenate(list_eeg, 2)
            
      
            
            if 'stft' in DATATYPE:
                list_stft = np.concatenate(list_stft, -1)
                stfts2[name] = list_stft
            
            if 'eeg' in DATATYPE:
                eegs2[name] = list_eeg
            
            
            
            if 'img' in DATATYPE:
                eeg_all_region = np.concatenate(list_img, 0)
                
                
                fig = plt.figure(clear=True)
                fig.patch.set_facecolor('black')
                amp = 200
                for ii in range(eeg_all_region.shape[0]):
                    jj = ii * amp + (ii // 4) * amp
                    plt.plot(eeg_all_region[ii, :] + jj, color='red', linewidth=0.5)
                plt.xlim(-10, eeg_all_region.shape[1] + 10)
                plt.ylim(-amp/2, eeg_all_region.shape[0] * amp + amp/2*5)
                plt.axis('off')
                
                
                byte_stream = io.BytesIO()
                plt.savefig(byte_stream, format='png', bbox_inches='tight')
                byte_stream.seek(0)
                img = Image.open(byte_stream)
                img = np.array(img)[:, :, :1]
                byte_stream.truncate()
                plt.close('all')
    
    
                img = np.concatenate((img, img, img), 2)
                img = np.array(tf.image.resize(img/255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32)
                img = img[:, :, 0:1]
                
    
                img = np.concatenate([img[0*IMG_HIGH:1*IMG_HIGH, :, :],
                                      img[1*IMG_HIGH:2*IMG_HIGH, :, :],
                                      img[2*IMG_HIGH:3*IMG_HIGH, :, :],
                                      img[3*IMG_HIGH:4*IMG_HIGH, :, :]], -1)
                
                img[:, :, 0] = -img[:, :, 0]
                img[:, :, 2] = -img[:, :, 2]
                
                img = np.reshape(img, (img.shape[0], img.shape[1], img.shape[2], 1))
                
                
                
                
                
                
                
                
                
                
                eeg_all_region2 = eeg_all_region[:, round(eeg_all_region.shape[1]*1/4):round(eeg_all_region.shape[1]*3/4)]
                fig = plt.figure(clear=True)
                fig.patch.set_facecolor('black')
                amp = 150
                for ii in range(eeg_all_region2.shape[0]):
                    jj = ii * amp + (ii // 4) * amp
                    plt.plot(eeg_all_region2[ii, :] + jj, color='red', linewidth=0.5)
                plt.xlim(-5, eeg_all_region2.shape[1] + 5)
                plt.ylim(-amp/2, eeg_all_region2.shape[0] * amp + amp/2*5)
                plt.axis('off')
                
                
                byte_stream = io.BytesIO()
                plt.savefig(byte_stream, format='png', bbox_inches='tight')
                byte_stream.seek(0)
                img2 = Image.open(byte_stream)
                img2 = np.array(img2)[:, :, :1]
                byte_stream.truncate()
                plt.close('all')
    
    
                img2 = np.concatenate((img2, img2, img2), 2)
                img2 = np.array(tf.image.resize(img2/255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32)
                img2 = img2[:, :, 0:1]
                
    
                img2 = np.concatenate([img2[0*IMG_HIGH:1*IMG_HIGH, :, :],
                                      img2[1*IMG_HIGH:2*IMG_HIGH, :, :],
                                      img2[2*IMG_HIGH:3*IMG_HIGH, :, :],
                                      img2[3*IMG_HIGH:4*IMG_HIGH, :, :]], -1)
                
                img2[:, :, 0] = -img2[:, :, 0]
                img2[:, :, 2] = -img2[:, :, 2]
                
                
                img2 = np.reshape(img2, (img2.shape[0], img2.shape[1], img2.shape[2], 1))
                
                
                
                
                
                
                eeg_all_region3 = eeg_all_region[:, round(eeg_all_region.shape[1]*2/5):round(eeg_all_region.shape[1]*3/5)]
                fig = plt.figure(clear=True)
                fig.patch.set_facecolor('black')
                amp = 100
                for ii in range(eeg_all_region3.shape[0]):
                    jj = ii * amp + (ii // 4) * amp
                    plt.plot(eeg_all_region3[ii, :] + jj, color='red', linewidth=0.5)
                plt.xlim(-2, eeg_all_region3.shape[1] + 2)
                plt.ylim(-amp/2, eeg_all_region3.shape[0] * amp + amp/2*5)
                plt.axis('off')
                
                
                byte_stream = io.BytesIO()
                plt.savefig(byte_stream, format='png', bbox_inches='tight')
                byte_stream.seek(0)
                img3 = Image.open(byte_stream)
                img3 = np.array(img3)[:, :, :1]
                byte_stream.truncate()
                plt.close('all')
    
    
                img3 = np.concatenate((img3, img3, img3), 2)
                img3 = np.array(tf.image.resize(img3/255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32)
                img3 = img3[:, :, 0:1]
                
    
                img3 = np.concatenate([img3[0*IMG_HIGH:1*IMG_HIGH, :, :],
                                      img3[1*IMG_HIGH:2*IMG_HIGH, :, :],
                                      img3[2*IMG_HIGH:3*IMG_HIGH, :, :],
                                      img3[3*IMG_HIGH:4*IMG_HIGH, :, :]], -1)
                
                img3[:, :, 0] = -img3[:, :, 0]
                img3[:, :, 2] = -img3[:, :, 2]
                
                img3 = np.reshape(img3, (img3.shape[0], img3.shape[1], img3.shape[2], 1))
                
                
                
                
                img = np.concatenate([img, img2, img3], -1)
                
                imgs2[name] = img


    
    # INFER EFFICIENTNET ON TEST
    preds = []
    model = build_model(TARGETS)
    test_gen = DataGenerator(test, shuffle=False, batch_size=BATCHSIZE*2, mode='test', specs = spectrograms2, eegs = eegs2, imgs=imgs2, stfts=stfts2, targets=TARGETS)
    
    for i in range(NSPLIT):
        print(f'Fold {i+1}')
        model.load_weights(os.path.join(LOAD_MODELS_FROM, f'f{i}_stage{STAGETEST}.h5'))
        pred = model.predict(test_gen, verbose=1)
        preds.append(pred)
    pred = np.mean(preds,axis=0)
    print()
    print('Test preds shape',pred.shape)
    
    
    sub = pd.DataFrame({'eeg_id':test.eeg_id.values})
    sub[TARGETS] = pred
    sub.to_csv('submission.csv',index=False)
    print('Submissionn shape',sub.shape)
    sub.head()
    
    
    # SANITY CHECK TO CONFIRM PREDICTIONS SUM TO ONE
    sub.iloc[:,-6:].sum(axis=1)










Looking in links: /kaggle/input/tf-efficientnet-whl-files
Processing /kaggle/input/tf-efficientnet-whl-files/efficientnet-1.1.1-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/tf-efficientnet-whl-files/efficientnet-1.1.1-py3-none-any.whl'

Looking in links: /kaggle/input/cupy20240401
ERROR: Could not find a version that satisfies the requirement cupy (from versions: none)
ERROR: No matching distribution found for cupy
['eeg', 'spe', 'img']


2026-04-28 04:37:48.152157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777351068.165716      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777351068.169776      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 04:37:48.184716: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version = 2.18.0
Using 0 GPU
Mixed precision enabled
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


2026-04-28 04:37:56.337224: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ModuleNotFoundError: No module named 'efficientnet'